# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Laspric/flyrank-internship-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## 1. My rule and its reason codes

I will prioritize a content page for refresh review when it is both:

1. **Stale:** it has not been updated for at least 91 days.
2. **Visible:** it has at least 100 impressions in the last 90 days.

A page that meets both conditions receives the highest baseline priority.

### Reason codes

* `stale_and_visible` — stale and meaningfully visible.
* `visible_but_not_stale` — visible but recently updated.
* `stale_but_low_visibility` — stale but has low visibility.
* `low_priority` — neither stale nor meaningfully visible.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

## 2. Build the ranked queue

I will apply the baseline rule to every content page, assign a transparent score, attach a reason code and action, rank the pages by priority, and save the resulting queue as `work/outputs/baseline_action_score.csv`.


In [24]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Staleness signal
df["stale"] = (
    df["days_since_last_update"] >= 91
).astype(int)

# Visibility signal
df["visible"] = (
    df["impressions_90d"] >= 100
).astype(int)

# Transparent baseline score
df["score"] = df["stale"] * df["visible"]

# Reason code
def assign_reason(row):
    if row["stale"] == 1 and row["visible"] == 1:
        return "stale_and_visible"
    elif row["stale"] == 0 and row["visible"] == 1:
        return "visible_but_not_stale"
    elif row["stale"] == 1 and row["visible"] == 0:
        return "stale_but_low_visibility"
    else:
        return "low_priority"

df["reason_code"] = df.apply(assign_reason, axis=1)

# Action
df["action"] = np.where(
    df["score"] == 1,
    "refresh_review",
    "monitor"
)

# Rank pages
queue = df.sort_values(
    by=["score", "impressions_90d"],
    ascending=[False, False]
).copy()

# Keep required columns
queue = queue[
    [
        "content_id",
        "client_id",
        "score",
        "reason_code",
        "action",
        "days_since_last_update",
        "impressions_90d"
    ]
]

# Save ranked queue
output_path = "work/outputs/baseline_action_score.csv"
queue.to_csv(output_path, index=False)

print("Queue shape:", queue.shape)
print("Saved to:", output_path)

Queue shape: (30000, 7)
Saved to: work/outputs/baseline_action_score.csv


# 3. Top-20 review
For each of the top 20: action, reason code, confidence note, and what would make it wrong.

## 3. Top-20 review

I reviewed the 20 highest-ranked pages from the baseline queue. For each page, I record the action, reason code, confidence note, and what could make the recommendation wrong.


In [25]:
# Select the top 20 pages
top20 = queue.head(20).copy()

# Explain why each page is in the queue
top20["why_its_there"] = (
    "Stale for "
    + top20["days_since_last_update"].astype(str)
    + " days and has "
    + top20["impressions_90d"].astype(str)
    + " impressions in the last 90 days."
)

# Confidence note
top20["confidence_note"] = (
    "Strong baseline match: both staleness and visibility conditions are met."
)

# What could make the recommendation wrong
top20["what_would_make_it_wrong"] = (
    "The page may still be accurate and may not need a refresh."
)

# Display the final review table
top20 = top20[
    [
        "content_id",
        "score",
        "reason_code",
        "action",
        "days_since_last_update",
        "impressions_90d",
        "why_its_there",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
]

print(top20.to_string(index=False))

          content_id  score       reason_code         action  days_since_last_update  impressions_90d                                                      why_its_there                                                          confidence_note                                   what_would_make_it_wrong
content_5fe46e04994d      1 stale_and_visible refresh_review                     104           517715 Stale for 104 days and has 517715 impressions in the last 90 days. Strong baseline match: both staleness and visibility conditions are met. The page may still be accurate and may not need a refresh.
content_2dba2b1f9536      1 stale_and_visible refresh_review                     104           443434 Stale for 104 days and has 443434 impressions in the last 90 days. Strong baseline match: both staleness and visibility conditions are met. The page may still be accurate and may not need a refresh.
content_2c2606c5d176      1 stale_and_visible refresh_review                     104           34

## 4. Weak picks + leakage check

The baseline can identify pages that are stale and visible, but it cannot confirm that a page actually needs a refresh. A page may still be accurate despite being old.

The baseline uses only `days_since_last_update` and `impressions_90d`. The decline label is used only for evaluation, and no label-derived columns are used in the baseline score.

The top-ranked pages should therefore be treated as decision-support recommendations rather than guaranteed declining pages.



In [26]:
# Weak-pick review and leakage check

print("Baseline features:")
print(["days_since_last_update", "impressions_90d"])

print("\nChecking for forbidden leakage columns...")

forbidden_columns = [
    "is_declining_label",
    "trend_direction",
    "trend_pct"
]

used_features = [
    "days_since_last_update",
    "impressions_90d"
]

leakage_found = [
    col for col in forbidden_columns
    if col in used_features
]

if len(leakage_found) == 0:
    print("PASS: No label-derived columns are used in the baseline score.")
else:
    print("FAIL: Leakage columns found:", leakage_found)

Baseline features:
['days_since_last_update', 'impressions_90d']

Checking for forbidden leakage columns...
PASS: No label-derived columns are used in the baseline score.


In [27]:
# Precision@K evaluation

df_eval = pd.read_csv("data/raw/content_refresh_anonymized.csv")

df_eval["is_declining_label"] = (
    df_eval["trend_direction"] == "down"
).astype(int)

# Match labels to the ranked queue
label_lookup = df_eval.set_index("content_id")["is_declining_label"]

queue_eval = queue.copy()
queue_eval["is_declining_label"] = (
    queue_eval["content_id"].map(label_lookup)
)

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

for k in [20, 50, 100]:
    precision = precision_at_k(
        queue_eval["score"],
        queue_eval["is_declining_label"],
        k
    )
    print(f"Precision@{k}: {precision:.4f} ({precision * 100:.2f}%)")

base_rate = queue_eval["is_declining_label"].mean()

print(f"\nOverall decline rate / base rate: {base_rate:.4f} ({base_rate * 100:.2f}%)")

Precision@20: 0.5500 (55.00%)
Precision@50: 0.4400 (44.00%)
Precision@100: 0.3800 (38.00%)

Overall decline rate / base rate: 0.5421 (54.21%)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.